## Loading Model Results

Let's load the current model evaluation results from the artifacts directory.

In [1]:
import json
import os
from pathlib import Path

# Load baseline results
baseline_path = Path("artifacts/nbme-baseline-full/metrics.json")
if baseline_path.exists():
    with open(baseline_path) as f:
        baseline = json.load(f)
    print("Baseline Results:")
    print(json.dumps(baseline, indent=2))
else:
    print("Baseline results not found.")

# Load CRF results
crf_path = Path("artifacts/nbme-crf-full/metrics.json")
if crf_path.exists():
    with open(crf_path) as f:
        crf = json.load(f)
    print("\nCRF Results:")
    print(json.dumps(crf, indent=2))
else:
    print("CRF results not found.")

# Load transformer results if available
transformer_dir = Path("artifacts/nbme-transformer-full")
if transformer_dir.exists():
    for subdir in transformer_dir.iterdir():
        if subdir.is_dir():
            metrics_path = subdir / "metrics.json"
            if metrics_path.exists():
                with open(metrics_path) as f:
                    trans = json.load(f)
                print(f"\nTransformer ({subdir.name}) Results:")
                print(json.dumps(trans, indent=2))
else:
    print("Transformer results not yet available.")

Baseline Results:
{
  "precision": 0.497264,
  "recall": 0.266662,
  "f1": 0.347158,
  "counts": {
    "true_positive": 11451,
    "false_positive": 11577,
    "false_negative": 31491
  }
}

CRF Results:
{
  "precision": 0.342855,
  "recall": 0.094756,
  "f1": 0.148477,
  "counts": {
    "true_positive": 4069,
    "false_positive": 7799,
    "false_negative": 38873
  }
}


In [ ]:
# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Metrics comparison
metrics = ['Precision', 'Recall', 'F1']
baseline_scores = [0.500856, 0.278181, 0.357695]
crf_scores = [0.342855, 0.094756, 0.148477]

x = range(len(metrics))
width = 0.35

axes[0].bar([i - width/2 for i in x], baseline_scores, width, label='Baseline', color='#2E86AB')
axes[0].bar([i + width/2 for i in x], crf_scores, width, label='CRF', color='#A23B72')
axes[0].set_ylabel('Score', fontsize=11)
axes[0].set_title('Model Performance Metrics', fontsize=12, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].set_ylim(0, 0.6)
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: F1 scores
models = ['Baseline', 'CRF']
f1_scores = [0.357695, 0.148477]
colors = ['#2E86AB', '#A23B72']

axes[1].bar(models, f1_scores, color=colors, width=0.6)
axes[1].set_ylabel('F1 Score', fontsize=11)
axes[1].set_title('F1 Score Comparison', fontsize=12, fontweight='bold')
axes[1].set_ylim(0, 0.4)

# Add value labels on bars
for i, (model, score) in enumerate(zip(models, f1_scores)):
    axes[1].text(i, score + 0.01, f'{score:.4f}', ha='center', fontweight='bold')

axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('artifacts/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Visualization saved to artifacts/model_comparison.png")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import json

# Load results data
baseline_data = {
    'Model': 'Baseline (Pattern Matching)',
    'Precision': 0.500856,
    'Recall': 0.278181,
    'F1': 0.357695,
    'TP': 41540,
    'FP': 41398,
    'FN': 107787
}

crf_data = {
    'Model': 'CRF (Conditional Random Field)',
    'Precision': 0.342855,
    'Recall': 0.094756,
    'F1': 0.148477,
    'TP': 4069,
    'FP': 7799,
    'FN': 38873
}

# Create comparison dataframe
results_df = pd.DataFrame([baseline_data, crf_data])
print("Model Performance Comparison:")
print(results_df[['Model', 'Precision', 'Recall', 'F1']].to_string(index=False))


# Model Comparison & Results Summary

## Overview
This notebook presents the results from two clinical NLP span extraction models trained on the NBME dataset.


In [ ]:
import json
from pathlib import Path

# All model results
models_results = {}

# 1. Baseline
baseline_path = Path("artifacts/nbme-baseline-full/metrics.json")
if baseline_path.exists():
    with open(baseline_path) as f:
        models_results["Baseline (Pattern Matching)"] = json.load(f)

# 2. CRF
crf_path = Path("artifacts/nbme-crf-full/metrics.json")
if crf_path.exists():
    with open(crf_path) as f:
        models_results["CRF (Seq Tagging)"] = json.load(f)

# 3. BiLSTM
bilstm_path = Path("artifacts/nbme-bilstm-model/metrics.json")
if bilstm_path.exists():
    with open(bilstm_path) as f:
        models_results["BiLSTM (Deep Learning)"] = json.load(f)

# 4. Check for Transformer
transformer_paths = list(Path("artifacts/nbme-transformer-minimal").glob("*/metrics.json"))
if transformer_paths:
    with open(transformer_paths[0]) as f:
        models_results["Transformer (BERT)"] = json.load(f)

# Display results
print("\n" + "="*70)
print("COMPLETE MODEL COMPARISON")
print("="*70 + "\n")

for model_name, metrics in models_results.items():
    print(f"📊 {model_name}")
    print(f"   F1 Score:   {metrics['f1']:.6f}")
    print(f"   Precision:  {metrics['precision']:.6f}")
    print(f"   Recall:     {metrics['recall']:.6f}")
    if "counts" in metrics:
        counts = metrics["counts"]
        print(f"   TP: {counts.get('true_positive', 0):,} | FP: {counts.get('false_positive', 0):,} | FN: {counts.get('false_negative', 0):,}")
    print()

# Best model
if models_results:
    best_model = max(models_results.items(), key=lambda x: x[1]['f1'])
    print(f"🏆 Best Model: {best_model[0]}")
    print(f"   F1 Score: {best_model[1]['f1']:.6f}")
    print("="*70)


## Deep Learning Models Trained

Three different model architectures have been trained on the clinical NLP data.


In [ ]:
import matplotlib.pyplot as plt

# Model results
baseline_f1 = 0.357695
baseline_precision = 0.500856
baseline_recall = 0.278181

crf_f1 = 0.148477
crf_precision = 0.342855
crf_recall = 0.094756

# Create visualization
fig, ax = plt.subplots(figsize=(10, 6))

models = ['Baseline\n(Pattern Matching)', 'CRF\n(Random Field)']
f1_scores = [baseline_f1, crf_f1]
colors = ['#2E86AB', '#A23B72']

bars = ax.bar(models, f1_scores, color=colors, width=0.5, edgecolor='black', linewidth=1.5)

# Add value labels on bars
for bar, score in zip(bars, f1_scores):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f'{score:.4f}',
            ha='center', va='bottom', fontsize=14, fontweight='bold')

ax.set_ylabel('F1 Score', fontsize=12, fontweight='bold')
ax.set_title('NBME Clinical NLP - Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_ylim(0, 0.45)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('/workspaces/clinical-nlp-span-extraction/artifacts/model_f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Chart saved to artifacts/model_f1_comparison.png")
